# DART 사업보고서 정보 추출 (LLM)

`dart_business_parser.ipynb`가 만든 회사별 입력 파일(`source/dart/business/inputs/`)을 LLM에 넣어 **제품·영어 검색어·매입처·고객·경쟁사·계약** 같은 해석이 필요한 정보를 뽑는다. 항목과 원칙은 `graphDB.md` 5장을 따른다.

- **실행 수단**(`graphDB.md` 4.4 LLM 실행 수단, 1절 `RUNNER`):
  - 지시문 다듬기: `'codex'`(ChatGPT 구독). 회사 하나마다 `codex exec`를 새로 실행한다. 결과 분석과 지시문 조정은 Claude Code가 한다.
  - API 확인: `'api'`(OpenAI API 일반 요청). 최종 지시문이 API에서도 Codex와 같은 결과를 내는지 몇십 개로 확인한다.
  - 전체 추출: `'batch'`(OpenAI 배치 API, 24시간 안 처리, 가격 50%). 제출한 뒤 다시 실행하면 결과를 받아 회사별로 나눠 저장한다.
  - `'claude'`(Claude Code 구독)로도 돌릴 수 있게 남겨 두었다.
- **전체 추출은 두 갈래로 동시에 돌린다**(`graphDB.md` 4.4). 요청이 큰 회사(`TARGET = 'long'`, 요청 약 1.5만 토큰 이상, 97개)는 구독 Codex로 차례로, 나머지(`TARGET = 'short'`)는 배치로 보낸다. 배치는 대기열 한도(Build 등급 300만 입력 토큰) 안에서 나눠 제출하고, 앞 배치가 끝나면 다음을 제출한다. 어느 쪽으로 돌렸든 같은 지시문·모델로 성공한 회사는 다시 돌리지 않는다.
- **원본 결과를 보관한다**(5.4). LLM이 돌려준 응답 전체를 회사별 파일로 남기고 고치지 않는다. 가공(이름 통일, 그래프)은 이 파일을 읽어서 한다.
- **실행은 꺼 두었다.** 1절 `RUN = True`로 바꿔야 실행된다. 실행은 구독 사용량을 쓰므로 건수와 모델을 정하고 돌린다.
- 이미 같은 입력·지시문·모델로 성공한 회사는 건너뛴다. 사용량 한도에 걸리면 멈추고, 다음에 다시 실행하면 남은 회사부터 한다.

**입력**

| 파일 | 내용 |
|---|---|
| `source/dart/business/inputs/종목코드.txt` | 회사별 입력 글 (`dart_business_parser`) |
| `source/dart/business/tables/*.csv` | 규칙으로 읽은 표. LLM 결과 점검에 쓴다 |
| `prompts/dart_business_버전.md` | 지시문 (1절 `PROMPT_VERSION`) |
| `prompts/dart_business_버전.schema.json` | 결과 JSON 스키마 |

**출력** (모두 `source/dart/business/extract/` 아래)

| 파일 | 내용 |
|---|---|
| `raw/지시문버전/실행도구-모델/종목코드_실행시각.json` | LLM 응답 전체(결과 JSON, 토큰 사용량 포함). 고치지 않는다. 예: `raw/v5/codex-gpt-6.1-terra/`, `raw/v5/api-gpt-6.1-terra/`(API 일반 요청과 배치가 함께 쌓인다) |
| `batches/`, `batches.csv` | 배치로 보낸 요청 파일과 배치 기록(배치 ID, 상태, 회사 수) |
| `runs.csv` | 실행 기록: 회사, 입력 해시, 지시문 버전, 실행 도구, 모델, 시각, 상태, 토큰 |
| `check_지시문버전_실행도구-모델.csv` | 회사별 점검 결과 |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `dart_business_parser` → **`dart_business_extractor`** → `dart_business_cleaner` |
| 외부 요청·비용 | 있음. `RUNNER`가 `api`·`batch`면 OpenAI API 크레딧, `codex`·`claude`면 구독 사용량. 실행 전에 2절의 예상 건수·비용을 보고 승인받는다 |
| 다시 실행 | 된다. 같은 지시문·모델로 성공한 회사는 건너뛴다. 배치는 실행할 때마다 결과를 받고 남은 회사를 새로 제출한다 |
| 멈추는 경우 | `RUN=False`(기본)면 예상만 보여 주고 보내지 않는다. 사용량 한도, 배치 제출 실패에서 멈춘다 |
| 처리 순서 | 설정 → 대상 고르기·예상 비용 → 실행(Codex·API·Claude) 또는 배치 제출·결과 받기 → 점검 → 결과 확인 |

## 1. 설정

In [1]:
# ── 설정: 지시문 버전, 실행 수단·모델, 대상, 배치 한도, 가격 ──
import hashlib
import json
import os
import re
import subprocess
import tempfile
import time
import datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import jsonschema
import pandas as pd

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
BUSINESS = ROOT / 'source' / 'dart' / 'business'
INPUTS = BUSINESS / 'inputs'
TABLES = BUSINESS / 'tables'
EXTRACT = BUSINESS / 'extract'
EXTRACT.mkdir(parents=True, exist_ok=True)
RUNS_PATH = EXTRACT / 'runs.csv'

PROMPT_VERSION = 'v5'
PROMPT_PATH = ROOT / 'prompts' / f'dart_business_{PROMPT_VERSION}.md'
SCHEMA_PATH = ROOT / 'prompts' / f'dart_business_{PROMPT_VERSION}.schema.json'
SCHEMA = json.loads(SCHEMA_PATH.read_text(encoding='utf-8'))

# ── 실행 설정 ──
RUNNER = 'api'            # 'codex'(ChatGPT 구독), 'api'(OpenAI API 일반 요청), 'batch'(OpenAI 배치 API), 'claude'(Claude Code 구독)
MODEL = 'gpt-6.1-terra'   # 지시문 다듬기는 Terra로 고정(graphDB.md 4.4 모델 비교). claude로 돌릴 때는 'sonnet' 등
EFFORT = 'medium'         # 추론 강도: low, medium, high
PARALLEL = 2              # 동시에 돌릴 수. 늘리면 빨라지지만 사용량 한도에 빨리 닿는다
TIMEOUT_SEC = 900         # 회사 하나에 기다리는 최대 시간

# 대상: 'test1'(1개 동작 확인), 'pilot'(시범), 'all'(전체), 'long'(요청이 큰 회사, Codex로), 'short'(나머지, 배치로), 또는 종목코드 목록
TARGET = 'all'
LONG_TOKENS = 15_000      # 요청 토큰이 이 값 이상이면 'long'. 1만이면 272개(Codex 주간 한도 약 82%), 1.5만이면 97개(약 42%)

# 배치 대기열 한도(입력 토큰). OpenAI Build 등급의 gpt-6.1-terra 값(Limits 화면). Free 등급이면 하루 90만이다
BATCH_QUEUE_TOKENS = 3_000_000
BATCH_FILL = 0.85         # 요청 토큰은 어림값이라 대기열의 85%까지만 채운다
PILOT = ['005930', '000660', '009150', '373220', '003670', '005380', '009540', '012450', '068270', '196170',
         '090430', '004370', '051910', '034020', '015760', '035420', '034730', '105560', '192820', '010120']

RUN = False               # True로 바꿔야 실행한다

if RUNNER == 'claude':
    # Claude Code는 스키마를 지시문 끝에 붙인다. --json-schema를 쓰면 결과를 글과 결과 도구로 두 번 써서 출력이 두 배가 된다(시험에서 확인)
    PROMPT = (PROMPT_PATH.read_text(encoding='utf-8')
              + '\n\n## 결과 형식\n\n아래 JSON 스키마에 맞는 JSON 객체 하나만 출력한다. 앞뒤에 설명, 코드 블록 표시(```)를 붙이지 않는다.\n\n'
              + json.dumps(SCHEMA, ensure_ascii=False))
    RUN_NAME = MODEL or 'claude-default'
elif RUNNER in ('api', 'batch'):
    # API는 구조화 출력(엄격 모드)으로 결과 형식을 정한다. 일반 요청과 배치는 같은 요청이라 결과를 같은 곳에 쌓는다
    PROMPT = PROMPT_PATH.read_text(encoding='utf-8')
    RUN_NAME = 'api-' + MODEL
else:
    # Codex는 --output-schema로 결과 형식을 정하므로 스키마를 지시문에 넣지 않는다
    PROMPT = (PROMPT_PATH.read_text(encoding='utf-8')
              + '\n\n## 결과 형식\n\n결과는 주어진 출력 스키마에 맞는 JSON 하나다. 파일을 읽거나 명령을 실행하지 말고, 아래 입력만 보고 답한다.')
    RUN_NAME = 'codex-' + (MODEL or 'default')

# OpenAI API 가격(100만 토큰당 입력, 캐시 입력, 출력. 2026-11-12 공식 가격표). 배치는 모두 50%
PRICES = {'gpt-6.1-sol': (2.00, 0.10, 10.00), 'gpt-6.1-terra': (1.00, 0.05, 5.00), 'gpt-6-luna': (0.10, 0.01, 0.50)}


def env_value(name):
    """환경변수 → 프로젝트 루트 .env → graph_db/.env 순서로 값을 찾는다. 값은 화면에 쓰지 않는다."""
    if os.environ.get(name):
        return os.environ[name]
    for env_file in (ROOT.parent / '.env', ROOT / '.env'):
        if env_file.exists():
            for line in env_file.read_text(encoding='utf-8-sig').splitlines():
                key, _, value = line.partition('=')
                if key.strip() == name and value.strip():
                    return value.strip().strip('"\'')
    return None


def api_cost(model, input_tokens, cached_tokens, output_tokens, batch=False):
    """API 요금(달러). 캐시로 읽은 입력은 캐시 가격으로 센다."""
    p_in, p_cache, p_out = PRICES[model]
    usd = ((input_tokens - cached_tokens) * p_in + cached_tokens * p_cache + output_tokens * p_out) / 1e6
    return usd / 2 if batch else usd

RAW = EXTRACT / 'raw' / PROMPT_VERSION / RUN_NAME
RAW.mkdir(parents=True, exist_ok=True)
print('실행 도구', RUNNER, '/ 모델', MODEL or '(기본)', '/ effort', EFFORT, '/ 지시문', f'{len(PROMPT):,}자', '/ 결과', RAW.relative_to(ROOT))

실행 도구 api / 모델 gpt-6.1-terra / effort medium / 지시문 4,013자 / 결과 source/dart/business/extract/raw/v5/api-gpt-6.1-terra


## 2. 대상 고르기

입력 파일의 해시(SHA-256 앞 16자)로 입력이 바뀌었는지 안다. 실행 기록(`runs.csv`)에서 **같은 지시문 버전·모델로 성공한 회사는 실행 수단과 관계없이 건너뛴다**(Codex로 끝낸 회사를 배치로 다시 보내지 않는다). 입력이 바뀌면(새 사업보고서) 다시 돌린다. 실제로 보낸 지시문·실행 도구·effort는 해시(`prompt_hash`)로 남긴다.

**긴 회사와 짧은 회사**: 요청 토큰(지시문·스키마 포함)을 입력 글자 수로 어림해(측정값으로 맞춘 식: 3,047 + 0.569 × 글자) `LONG_TOKENS` 이상이면 `long`, 아니면 `short`로 나눈다.

In [2]:
# ── 대상 고르기: 이미 끝난 회사·배치를 기다리는 회사를 빼고, 예상 토큰·비용을 보여 준다 ──
def input_hash(text):
    """입력 글의 짧은 해시. 입력 파일이 바뀌면 다시 돌린다."""
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]


# 실행 도구가 claude일 때는 예전 기록과 같은 해시가 나오도록 실행 도구 이름을 넣지 않는다
PROMPT_HASH = input_hash(PROMPT + (MODEL or 'sonnet') + EFFORT) if RUNNER == 'claude' else input_hash(PROMPT + RUN_NAME + EFFORT)
RUN_COLUMNS = ['stock_code', 'input_hash', 'prompt_version', 'prompt_hash', 'runner', 'model', 'effort', 'run_utc', 'raw_file',
               'status', 'error', 'duration_s', 'input_tokens', 'cache_creation_tokens', 'cache_read_tokens', 'output_tokens',
               'reasoning_tokens', 'cost_usd_equiv']
runs = pd.read_csv(RUNS_PATH, dtype=str) if RUNS_PATH.exists() else pd.DataFrame(columns=RUN_COLUMNS)
if RUNS_PATH.exists() and list(runs.columns) != RUN_COLUMNS:
    # 열이 바뀌었으면(예: 실행 도구 열 추가) 새 열 구성으로 옮겨 쓴다. 그래야 아래에서 한 줄씩 붙일 때 열이 맞는다
    runs = runs.reindex(columns=RUN_COLUMNS)
    runs['runner'] = runs['runner'].fillna('claude')   # 실행 도구 열이 생기기 전 기록은 Claude Code로 돌린 것이다
    runs.to_csv(RUNS_PATH, index=False, encoding='utf-8-sig')
runs = runs.reindex(columns=RUN_COLUMNS)

all_codes = sorted(p.stem for p in INPUTS.glob('*.txt') if re.fullmatch(r'[0-9A-Z]{6}', p.stem))   # 동기화 사본 제외


def est_request_tokens(chars):
    """API 요청 하나의 입력 토큰 어림값(지시문·스키마 포함). v5로 잰 4개 회사의 실제 값으로 맞춘 식이다."""
    return 3047 + 0.569 * chars


req_tokens = {c: est_request_tokens(len((INPUTS / f'{c}.txt').read_text(encoding='utf-8'))) for c in all_codes}
long_codes = sorted((c for c in all_codes if req_tokens[c] >= LONG_TOKENS), key=lambda c: -req_tokens[c])   # 큰 것부터
short_codes = sorted((c for c in all_codes if req_tokens[c] < LONG_TOKENS), key=lambda c: req_tokens[c])
if TARGET == 'test1':
    wanted = PILOT[:1]
elif TARGET == 'pilot':
    wanted = PILOT
elif TARGET == 'all':
    wanted = all_codes
elif TARGET == 'long':
    wanted = long_codes
elif TARGET == 'short':
    wanted = short_codes
else:
    wanted = list(TARGET)

# 같은 지시문 버전·모델로 성공한 회사는 실행 수단(Codex, API, 배치)과 관계없이 끝난 것으로 본다
done = set(runs.loc[(runs['status'] == 'ok') & (runs['prompt_version'] == PROMPT_VERSION) & (runs['model'] == MODEL),
                    ['stock_code', 'input_hash']].itertuples(index=False, name=None))
todo = [c for c in wanted if (c, input_hash((INPUTS / f'{c}.txt').read_text(encoding='utf-8'))) not in done]
# 배치로 제출해 결과를 기다리는 회사는 다시 보내지 않는다(3-1절)
BATCH_DIR = EXTRACT / 'batches'
BATCH_LOG = EXTRACT / 'batches.csv'
waiting = set()
if BATCH_LOG.exists():
    blog = pd.read_csv(BATCH_LOG, dtype=str).fillna('')
    for b in blog[blog['collected'] != 'yes'].itertuples():
        waiting |= {json.loads(l)['custom_id'].split('|')[0] for l in (BATCH_DIR / b.input_file).read_text(encoding='utf-8').splitlines()}
todo = [c for c in todo if c not in waiting]

sizes = {c: len((INPUTS / f'{c}.txt').read_text(encoding='utf-8')) for c in todo}
print(f'대상 {len(wanted)}개 / 이미 끝남 {len(wanted) - len(todo) - len(waiting & set(wanted))}개 / 배치 결과 기다림 {len(waiting & set(wanted))}개'
      f' / 이번에 돌릴 것 {len(todo)}개, 입력 {sum(sizes.values()):,}자')

# 예상 비용(API·배치): 입력은 위의 요청 토큰 어림값, 출력은 시험에서 잰 Terra 비율(입력 길이에 비례)로 어림한다. ±30% 정도 차이가 날 수 있다
if RUNNER in ('api', 'batch') and todo:
    est_in = sum(req_tokens[c] for c in todo)
    est_out = sum(max(800, 0.5 * n - 900) for n in sizes.values())
    est = api_cost(MODEL, est_in, 0, est_out, batch=(RUNNER == 'batch'))
    print(f'예상 비용({RUNNER}, {MODEL}): 입력 약 {est_in / 1e6:.2f}백만 토큰, 출력 약 {est_out / 1e6:.2f}백만 토큰 → 약 ${est:.2f}')

대상 1063개 / 이미 끝남 1063개 / 배치 결과 기다림 0개 / 이번에 돌릴 것 0개, 입력 0자


## 3. 실행

회사 하나마다 실행 도구를 새로 띄운다. 프로젝트 설정·메모리가 섞이지 않게 빈 임시 폴더에서 실행한다.

**Codex** (`codex exec`): 지시문과 입력 글을 표준입력으로 함께 넘긴다.

| 옵션 | 이유 |
|---|---|
| `--output-schema` | 결과를 스키마에 맞는 JSON으로 받는다 |
| `--json`, `-o` | 진행 기록(토큰 수 포함)과 최종 결과를 받는다 |
| `-s read-only`, `--ephemeral`, `--ignore-user-config`, `--ignore-rules` | 파일을 고칠 수 없게 하고, 세션 기록·사용자 설정·규칙 파일을 쓰지 않는다 |
| `-m`, `-c model_reasoning_effort` | 처음부터 끝까지 같은 모델·추론 강도로 돌린다 |

**OpenAI API** (`'api'`): Responses API에 지시문(`instructions`)과 입력 글(`input`)을 보내고, 결과 형식은 구조화 출력(`json_schema`, 엄격 모드)으로 정한다. 추론 강도는 `reasoning.effort`, 응답 저장은 끈다(`store=False`). 키는 `.env`의 `OPENAI_API_KEY`에서 읽고 화면·파일에 쓰지 않는다. 실제 요금을 공식 가격으로 계산해 실행 기록(`cost_usd_equiv`)에 남긴다.

**Claude Code** (`claude -p`): 기본 지시문 대신 우리 지시문(`--system-prompt`), 도구·외부 연결·스킬 끄기, `--output-format json`, `--no-session-persistence`.

- Codex·Claude Code로 돌릴 때는 환경변수 `OPENAI_API_KEY`·`ANTHROPIC_API_KEY`가 있으면 구독 대신 API 요금으로 돌 수 있어 실행할 때 지운다.
- 응답은 성공·실패와 관계없이 `raw/`에 그대로 저장하고, 회사 하나가 끝날 때마다 `runs.csv`에 바로 한 줄 남긴다. 사용량 한도 오류가 나면 남은 회사는 돌리지 않고 멈춘다.

In [3]:
# ── 실행(배치가 아닐 때): 회사마다 실행 도구를 새로 띄우거나 API 요청을 보낸다 ──
RUN_CWD = Path(tempfile.gettempdir()) / 'graphdb_llm_run'
RUN_CWD.mkdir(exist_ok=True)
# 구독 실행에 API 키가 넘어가지 않게 뺀다
ENV = {k: v for k, v in os.environ.items() if k not in ('ANTHROPIC_API_KEY', 'ANTHROPIC_AUTH_TOKEN', 'OPENAI_API_KEY')}
LIMIT_RE = re.compile(r'usage limit|rate limit|limit reached|quota|too many requests|429', re.I)


def command(out_file):
    """회사 하나를 돌리는 명령(Codex: 읽기 전용·사용자 설정 무시·출력 스키마, Claude: 도구 없이 JSON만)."""
    if RUNNER == 'claude':
        return (['claude', '-p'] + (['--model', MODEL] if MODEL else []) + ['--effort', EFFORT, '--system-prompt', PROMPT,
                '--output-format', 'json', '--no-session-persistence', '--tools', '', '--strict-mcp-config', '--disable-slash-commands'])
    return (['codex', 'exec', '--json', '--ephemeral', '--skip-git-repo-check', '-s', 'read-only',
             '--ignore-user-config', '--ignore-rules', '-C', str(RUN_CWD),
             '--output-schema', str(SCHEMA_PATH), '-o', str(out_file), '-c', f'model_reasoning_effort="{EFFORT}"']
            + (['-m', MODEL] if MODEL else []) + ['-'])


def api_output_text(resp):
    """Responses API 응답에서 결과 글을 꺼낸다."""
    for item in resp.get('output') or []:
        if item.get('type') == 'message':
            for c in item.get('content') or []:
                if c.get('type') == 'output_text':
                    return c.get('text') or ''
    return ''


def parse_result(resp):
    """원본 응답에서 결과 JSON을 꺼낸다(API: output, Codex: 최종 메시지, Claude: result)."""
    if 'custom_id' in resp:                       # 배치 결과 한 줄: 응답 본문이 한 겹 더 싸여 있다
        resp = (resp.get('response') or {}).get('body') or {}
    if isinstance(resp.get('structured_output'), dict):
        return resp['structured_output']
    if isinstance(resp.get('output'), list):
        text = api_output_text(resp)
    else:
        text = (resp.get('last_message') if 'last_message' in resp else resp.get('result')) or ''
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    if not text.startswith('{'):
        text = text[text.find('{'):]
    return json.loads(text)


def codex_response(proc, out_file):
    """codex exec의 JSONL 진행 기록과 최종 메시지를 한 덩어리로 묶는다."""
    events = []
    for line in proc.stdout.splitlines():
        try:
            events.append(json.loads(line))
        except json.JSONDecodeError:
            events.append({'type': 'text', 'text': line})
    last = out_file.read_text(encoding='utf-8') if out_file.exists() else ''
    usage = {}
    for e in events:
        if e.get('type') == 'turn.completed' and isinstance(e.get('usage'), dict):
            usage = e['usage']
    errors = [e for e in events if e.get('type') in ('error', 'turn.failed')]
    return {'runner': 'codex', 'model': MODEL, 'returncode': proc.returncode, 'usage': usage, 'last_message': last,
            'events': events, 'stderr': proc.stderr[-3000:], 'is_error': bool(errors) or proc.returncode != 0 or not last}


mask = lambda s: re.sub(r'org-[A-Za-z0-9]{6,}', 'org-***', re.sub(r'sk-[A-Za-z0-9_\-]{6,}', 'sk-***', str(s)))   # 오류 문구의 키·조직 ID를 가린다
_client = None


def openai_client():
    """OpenAI 클라이언트를 한 번만 만든다. 키는 .env에서 읽는다."""
    global _client
    if _client is None:
        import openai
        key = env_value('OPENAI_API_KEY')
        assert key, '.env에 OPENAI_API_KEY가 없다.'
        _client = openai.OpenAI(api_key=key, max_retries=3)
    return _client


def api_body(text):
    """Responses API 요청 본문. 일반 요청과 배치가 같은 본문을 쓴다."""
    return dict(model=MODEL, instructions=PROMPT, input=text, reasoning={'effort': EFFORT}, store=False,
                text={'format': {'type': 'json_schema', 'name': 'dart_business', 'schema': SCHEMA, 'strict': True}})


def finish(row, resp, batch=False):
    """API 응답의 토큰·비용을 적고 결과를 검사한다."""
    u = resp.get('usage') or {}
    cached = (u.get('input_tokens_details') or {}).get('cached_tokens') or 0
    row.update(input_tokens=u.get('input_tokens'), cache_read_tokens=cached, output_tokens=u.get('output_tokens'),
               reasoning_tokens=(u.get('output_tokens_details') or {}).get('reasoning_tokens'))
    if u.get('input_tokens') is not None and MODEL in PRICES:
        row['cost_usd_equiv'] = round(api_cost(MODEL, u['input_tokens'], cached, u.get('output_tokens') or 0, batch), 5)
    if resp.get('status') != 'completed':
        row.update(status='error', error=mask(json.dumps(resp.get('incomplete_details') or resp.get('error') or resp.get('status')))[:300])
        return row
    try:
        out = parse_result(resp)
        errors = [e.message for e in jsonschema.Draft202012Validator(SCHEMA).iter_errors(out)]
        row.update(status='ok' if not errors else 'bad_schema', error='; '.join(errors)[:300])
    except (json.JSONDecodeError, TypeError) as e:
        row.update(status='bad_json', error=str(e)[:300])
    return row


def run_api(code_, text, row):
    """OpenAI API 일반 요청으로 회사 하나를 돌린다."""
    import openai
    t0 = time.time()
    try:
        resp = openai_client().responses.create(**api_body(text), timeout=TIMEOUT_SEC).model_dump()
    except openai.RateLimitError as e:          # 요청 한도 또는 잔액 부족
        # 요청 하나가 분당 한도보다 크면 다시 해도 안 되므로 따로 표시하고 다음 회사로 넘어간다
        status = 'too_large' if 'Request too large' in str(e) else 'limit'
        return row | dict(status=status, error=mask(e)[:300], duration_s=round(time.time() - t0, 1))
    except openai.APIError as e:
        return row | dict(status='error', error=mask(e)[:300], duration_s=round(time.time() - t0, 1))
    row['duration_s'] = round(time.time() - t0, 1)
    (RAW / row['raw_file']).write_text(json.dumps(resp, ensure_ascii=False, indent=1), encoding='utf-8')
    return finish(row, resp)


def run_one(code_):
    """회사 하나를 실행 수단에 맞게 돌리고 원본 응답을 저장한 뒤 실행 기록 한 줄을 돌려준다."""
    text = (INPUTS / f'{code_}.txt').read_text(encoding='utf-8')
    now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
    raw_file = f"{code_}_{now.strftime('%Y%m%dT%H%M%S')}.json"     # 시각을 붙여 이전 결과를 덮어쓰지 않는다
    out_file = RUN_CWD / f'{code_}_{now.strftime("%H%M%S")}.out'
    row = dict(stock_code=code_, input_hash=input_hash(text), prompt_version=PROMPT_VERSION, prompt_hash=PROMPT_HASH, runner=RUNNER,
               model=MODEL or 'default', effort=EFFORT, run_utc=now.isoformat(), raw_file=raw_file, status='', error='')
    if RUNNER == 'api':
        return run_api(code_, text, row)
    stdin = text if RUNNER == 'claude' else PROMPT + '\n\n---\n\n# 입력\n\n' + text
    t0 = time.time()
    try:
        proc = subprocess.run(command(out_file), input=stdin, capture_output=True, text=True,
                              timeout=TIMEOUT_SEC, cwd=RUN_CWD, env=ENV)
    except subprocess.TimeoutExpired:
        return row | dict(status='timeout', duration_s=round(time.time() - t0, 1))
    row['duration_s'] = round(time.time() - t0, 1)
    if RUNNER == 'codex':
        resp = codex_response(proc, out_file)
        out_file.unlink(missing_ok=True)
        u = resp['usage']
        row.update(input_tokens=u.get('input_tokens'), cache_read_tokens=u.get('cached_input_tokens'),
                   output_tokens=u.get('output_tokens'), reasoning_tokens=u.get('reasoning_output_tokens'))
    else:
        try:
            resp = json.loads(proc.stdout)
        except json.JSONDecodeError:
            resp = {'is_error': True, 'result': proc.stdout[-2000:], 'stderr': proc.stderr[-2000:]}
        u = resp.get('usage') or {}
        row.update(input_tokens=u.get('input_tokens'), cache_creation_tokens=u.get('cache_creation_input_tokens'),
                   cache_read_tokens=u.get('cache_read_input_tokens'), output_tokens=u.get('output_tokens'),
                   reasoning_tokens=(u.get('output_tokens_details') or {}).get('thinking_tokens'), cost_usd_equiv=resp.get('total_cost_usd'))
    (RAW / raw_file).write_text(json.dumps(resp, ensure_ascii=False, indent=1), encoding='utf-8')
    if resp.get('is_error') or proc.returncode != 0:
        msg = ' '.join(str(x) for x in [resp.get('result', ''), resp.get('stderr', ''), proc.stderr,
                                         [e for e in resp.get('events', []) if e.get('type') in ('error', 'turn.failed')]])
        row.update(status='limit' if LIMIT_RE.search(msg) else 'error', error=msg.strip()[:300])
        return row
    try:
        out = parse_result(resp)
        errors = [e.message for e in jsonschema.Draft202012Validator(SCHEMA).iter_errors(out)]
        row.update(status='ok' if not errors else 'bad_schema', error='; '.join(errors)[:300])
    except (json.JSONDecodeError, TypeError) as e:
        row.update(status='bad_json', error=str(e)[:300])
    return row


new_rows = []
if RUN and todo and RUNNER != 'batch':
    print(f'{len(todo)}개 실행 시작 (동시 {PARALLEL}개)')
    stop = False
    with ThreadPoolExecutor(PARALLEL) as pool:
        pending = {pool.submit(run_one, c): c for c in todo[:PARALLEL]}
        queue = list(todo[PARALLEL:])
        while pending:
            for fut in as_completed(list(pending)):
                c = pending.pop(fut)
                row = fut.result()
                new_rows.append(row)
                # 끝날 때마다 바로 기록한다(중간에 멈추거나 한도에 걸려도 끝난 회사는 남는다)
                pd.DataFrame([row]).reindex(columns=RUN_COLUMNS).to_csv(
                    RUNS_PATH, mode='a', header=not RUNS_PATH.exists(), index=False, encoding='utf-8-sig')
                print(f"  {c} {row['status']:8} {row.get('duration_s', '')}초 입력 {row.get('input_tokens')} 출력 {row.get('output_tokens')}")
                if row['status'] == 'limit':
                    stop = True
                if queue and not stop:
                    nxt = queue.pop(0)
                    pending[pool.submit(run_one, nxt)] = nxt
                break
    if stop:
        print('사용량 한도에 걸려 멈췄다. 한도가 풀린 뒤 다시 실행하면 남은 회사부터 한다.')
    runs = pd.read_csv(RUNS_PATH, dtype=str).reindex(columns=RUN_COLUMNS)
    runs['runner'] = runs['runner'].fillna('claude')
elif not RUN:
    print('RUN = False: 실행하지 않았다.')

RUN = False: 실행하지 않았다.


## 3-1. 배치 제출과 결과 받기

`RUNNER = 'batch'`일 때만 쓴다. OpenAI 배치 API는 요청을 파일 하나로 모아 올리면 24시간 안에 처리하고, 가격은 일반 요청의 50%다.

1. **제출** (`RUN = True`): 대기열에 남은 자리(`BATCH_QUEUE_TOKENS`의 85%에서 기다리는 배치의 요청 토큰을 뺀 양)만큼 회사를 골라 `batches/배치이름.jsonl`로 만들어 올리고 배치를 만든다. 자리가 없으면 제출하지 않는다. 요청 본문은 일반 요청(`'api'`)과 같다. 배치 ID를 `batches.csv`에 남긴다.
2. **결과 받기**: 노트북을 다시 실행하면 기다리는 배치의 상태를 확인하고, 끝났으면 결과를 받아 회사별 원본 결과(`raw/`)로 나누고 실행 기록(`runs.csv`)에 남긴다. 상태 확인과 결과 받기는 요금이 들지 않는다.

- 결과를 기다리는 회사는 2절에서 대상에서 빠지므로 두 번 제출되지 않는다.
- **순서**: 실행할 때마다 ① 기다리는 배치의 상태를 확인하고 끝났으면 결과를 받는다 ② 방금 받은 결과까지 반영해 남은 회사를 다시 계산한다 ③ 대기열에 남은 자리만큼 회사를 고른다 ④ 제출한다(`RUN = True`일 때). 그래서 노트북을 다시 실행하기만 하면 짧은 회사 966개(약 760만 토큰)가 대략 3번에 나눠 제출된다.
- 실패한 요청은 실행 기록에 `error`로 남고, 다음 제출 때 다시 대상이 된다.

In [4]:
# ── 배치(RUNNER='batch'): 결과 받기 → 남은 회사 계산 → 대기열 자리만큼 제출 ──
def batch_log():
    """배치 기록(batches.csv)을 읽는다."""
    return pd.read_csv(BATCH_LOG, dtype=str).fillna('') if BATCH_LOG.exists() else pd.DataFrame(
        columns=['batch_id', 'created_utc', 'input_file', 'n', 'est_tokens', 'prompt_version', 'prompt_hash', 'model', 'effort', 'status', 'collected'])


# ① 기다리는 배치의 상태 확인과 결과 받기(요금 없음)
if RUNNER == 'batch' and BATCH_LOG.exists():
    log = batch_log()
    for i, b in log[log['collected'] != 'yes'].iterrows():
        info = openai_client().batches.retrieve(b['batch_id'])
        log.loc[i, 'status'] = info.status
        print(f"{b['batch_id']}: {info.status} / 완료 {info.request_counts.completed} / 실패 {info.request_counts.failed} / 전체 {info.request_counts.total}")
        if info.status not in ('completed', 'failed', 'expired', 'cancelled'):
            continue
        rows = []
        for file_id in [info.output_file_id, info.error_file_id]:
            if not file_id:
                continue
            for line in openai_client().files.content(file_id).text.splitlines():
                if not line.strip():
                    continue
                item = json.loads(line)
                code_, ihash = item['custom_id'].split('|')
                body = (item.get('response') or {}).get('body') or {}
                now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
                row = dict(stock_code=code_, input_hash=ihash, prompt_version=b['prompt_version'], prompt_hash=b['prompt_hash'],
                           runner='batch', model=b['model'], effort=b['effort'], run_utc=now.isoformat(),
                           raw_file=f"{code_}_{now.strftime('%Y%m%dT%H%M%S')}_batch.json", status='', error='')
                (RAW / row['raw_file']).write_text(json.dumps(item, ensure_ascii=False, indent=1), encoding='utf-8')
                if item.get('error') or not body:
                    row.update(status='error', error=mask(json.dumps(item.get('error') or item.get('response')))[:300])
                else:
                    row = finish(row, body, batch=True)
                rows.append(row)
        if info.status == 'failed' and info.errors:          # 배치 전체가 거절된 경우(예: 대기열 한도 초과)
            print('  배치 실패:', mask(info.errors)[:300])
        if rows:
            pd.DataFrame(rows).reindex(columns=RUN_COLUMNS).to_csv(RUNS_PATH, mode='a', header=not RUNS_PATH.exists(),
                                                                     index=False, encoding='utf-8-sig')
            print(f'  결과 {len(rows)}개 저장: 성공', sum(r['status'] == 'ok' for r in rows))
        log.loc[i, 'collected'] = 'yes'
    log.to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
    runs = pd.read_csv(RUNS_PATH, dtype=str).reindex(columns=RUN_COLUMNS)

# ② 방금 받은 결과까지 반영해 남은 회사를 다시 계산한다
if RUNNER == 'batch':
    done = set(runs.loc[(runs['status'] == 'ok') & (runs['prompt_version'] == PROMPT_VERSION) & (runs['model'] == MODEL),
                        ['stock_code', 'input_hash']].itertuples(index=False, name=None))
    log = batch_log()
    waiting = set()
    for b in log[log['collected'] != 'yes'].itertuples():
        waiting |= {json.loads(l)['custom_id'].split('|')[0] for l in (BATCH_DIR / b.input_file).read_text(encoding='utf-8').splitlines()}
    todo = [c for c in wanted if c not in waiting
            and (c, input_hash((INPUTS / f'{c}.txt').read_text(encoding='utf-8'))) not in done]

    # ③ 대기열에 남은 자리만큼 고른다(기다리는 배치의 요청 토큰은 대기열을 차지한다)
    queued = pd.to_numeric(log.loc[log['collected'] != 'yes', 'est_tokens'], errors='coerce').fillna(0).sum()
    room = BATCH_QUEUE_TOKENS * BATCH_FILL - queued
    pick, used = [], 0
    for c in todo:
        if used + req_tokens[c] > room:
            break
        pick.append(c)
        used += req_tokens[c]
    print(f'남은 회사 {len(todo)}개 / 기다리는 배치 약 {queued / 1e6:.2f}백만 토큰 / 넣을 수 있는 자리 약 {max(room, 0) / 1e6:.2f}백만 토큰'
          f' → 이번 제출 {len(pick)}개(약 {used / 1e6:.2f}백만 토큰, 예상 ${api_cost(MODEL, used, 0, sum(max(800, 0.5 * len((INPUTS / f"{c}.txt").read_text(encoding="utf-8")) - 900) for c in pick), batch=True):.2f})')

    # ④ 제출
    if RUN and pick:
        BATCH_DIR.mkdir(exist_ok=True)
        now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
        name = f"{PROMPT_VERSION}_{MODEL}_{now.strftime('%Y%m%dT%H%M%S')}.jsonl"
        lines = []
        for c in pick:
            text = (INPUTS / f'{c}.txt').read_text(encoding='utf-8')
            lines.append(json.dumps({'custom_id': f'{c}|{input_hash(text)}', 'method': 'POST', 'url': '/v1/responses',
                                     'body': api_body(text)}, ensure_ascii=False))
        (BATCH_DIR / name).write_text('\n'.join(lines) + '\n', encoding='utf-8')
        client = openai_client()
        with open(BATCH_DIR / name, 'rb') as f:
            uploaded = client.files.create(file=f, purpose='batch')
        batch = client.batches.create(input_file_id=uploaded.id, endpoint='/v1/responses', completion_window='24h',
                                      metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL})
        log = pd.concat([log, pd.DataFrame([dict(batch_id=batch.id, created_utc=now.isoformat(), input_file=name, n=str(len(pick)),
                         est_tokens=str(int(used)), prompt_version=PROMPT_VERSION, prompt_hash=PROMPT_HASH, model=MODEL, effort=EFFORT,
                         status=batch.status, collected='')])])
        log.to_csv(BATCH_LOG, index=False, encoding='utf-8-sig')
        print(f'배치 제출: {batch.id} / {len(pick)}개 / 요청 파일 {name}')

## 4. 점검

성공한 결과마다 아래를 본다. 규칙으로 읽은 표(`dart_business_parser`)를 기준으로 쓴다.

| 점검 | 기준 |
|---|---|
| 형식 | 결과가 JSON 스키마에 맞는가 |
| 근거 | 근거가 입력 글에 그대로 있는가(공백은 무시하고 비교). 없으면 LLM이 지어냈거나 고쳐 쓴 것이다. v3부터는 근거를 결과 맨 앞 `evidence` 목록에 한 번만 적고 항목은 번호(`evidence_id`)로 가리킨다 |
| 근거 번호 | 항목이 가리킨 번호가 근거 목록에 있는가(`bad_evidence_id`), 항목 이름이 가리킨 근거 안에 나오는가(`name_in_evidence`, 대략적인 비율). 번호를 잘못 붙였는지 본다 |
| 익명 | "A사", "B사" 같은 익명 상대가 들어갔는가 |
| 매입처 | 원재료 표에 실명 매입처가 있는 회사에서, 표의 매입처를 LLM 결과가 얼마나 담았는가 |

In [5]:
# ── 점검: 형식, 근거 문장이 원문에 있는지, 규칙으로 읽은 표와 맞는지 ──
EVIDENCE_KEYS = ['products', 'brands', 'materials', 'customers', 'competitors', 'market_shares', 'contracts',
                 'price_items', 'other_relations']
ANON_RE = re.compile(r'^[A-Z가-하]\s?사$|^[A-Z]社$|익명|국내\s*대기업|글로벌\s*.*업체')
squash = lambda s: re.sub(r'\s+', '', s or '')

materials_tbl = pd.read_csv(TABLES / 'materials.csv', dtype=str).fillna('')
validator = jsonschema.Draft202012Validator(SCHEMA)

checks = []
ok_runs = runs[(runs['status'] == 'ok') & (runs['prompt_hash'] == PROMPT_HASH)]
ok_runs = ok_runs.drop_duplicates('stock_code', keep='last')
for r in ok_runs.itertuples():
    resp = json.loads((RAW / r.raw_file).read_text(encoding='utf-8'))
    out = parse_result(resp)
    src = squash((INPUTS / f'{r.stock_code}.txt').read_text(encoding='utf-8'))
    schema_errors = [e.message for e in validator.iter_errors(out)]
    ev_map = {e['id']: e['text'] for e in out.get('evidence', [])}          # v3: 근거 목록
    items = [(k, x) for k in EVIDENCE_KEYS for x in out.get(k, [])]
    evid = [(k, x['evidence'] if 'evidence' in x else ev_map.get(x.get('evidence_id'), '')) for k, x in items]
    missing = [(k, e) for k, e in evid if not e or squash(e) not in src]
    bad_id = sum(1 for _, x in items if 'evidence_id' in x and x['evidence_id'] not in ev_map)
    # 항목 이름이 가리킨 근거 안에 있는가(번호를 잘못 붙였는지 보는 대략적인 점검). 괄호 앞부분만 비교한다
    item_name = lambda k, x: x.get('name') or x.get('item') or x.get('counterparty') or x.get('product') or ''
    name_in = [squash(re.split(r'[(（]', item_name(k, x))[0])[:12] in squash(e) for (k, x), (_, e) in zip(items, evid) if item_name(k, x)]
    names = ([x['name'] for x in out.get('customers', [])] + [x['name'] for x in out.get('competitors', [])]
             + [x['counterparty'] for x in out.get('contracts', [])] + [s['name'] for m in out.get('materials', []) for s in m['suppliers']])
    table_sup = {s.strip() for v in materials_tbl.loc[materials_tbl['stock_code'] == r.stock_code, 'suppliers'] for s in v.split(';') if s.strip()}
    llm_sup = {s['name'] for m in out.get('materials', []) for s in m['suppliers']}
    checks.append(dict(
        stock_code=r.stock_code, schema_errors=len(schema_errors),
        products=len(out.get('products', [])), products_body=sum(p['found_in'] == '본문' for p in out.get('products', [])),
        suppliers=len(llm_sup),
        customers=len(out.get('customers', [])), competitors=len(out.get('competitors', [])),
        contracts=len(out.get('contracts', [])), other=len(out.get('other_relations', [])),
        evidence_items=len(evid), evidence_missing=len(missing), bad_evidence_id=bad_id,
        name_in_evidence=round(sum(name_in) / max(len(name_in), 1), 2),
        evidence_unique=len(out['evidence']) if 'evidence' in out else len(set(e for _, e in evid)),
        anonymous=sum(bool(ANON_RE.search(n.strip())) for n in names),
        table_suppliers=len(table_sup),
        table_suppliers_covered=sum(any(squash(t) in squash(l) or squash(l) in squash(t) for l in llm_sup) for t in table_sup),
        output_tokens=r.output_tokens, notes=out.get('notes', '')))

check = pd.DataFrame(checks)
if len(check):
    check.insert(1, 'stock_name', check['stock_code'].map(
        pd.read_csv(ROOT / 'source' / 'dart' / 'parsed' / 'companies.csv', dtype=str).set_index('stock_code')['stock_name']))
    check.to_csv(EXTRACT / f'check_{PROMPT_VERSION}_{RUN_NAME}.csv', index=False, encoding='utf-8-sig')
print('점검한 회사', len(check))
check

점검한 회사 965


,stock_code,stock_name,schema_errors,products,products_body,suppliers,customers,competitors,contracts,other,evidence_items,evidence_missing,bad_evidence_id,name_in_evidence,evidence_unique,anonymous,table_suppliers,table_suppliers_covered,output_tokens,notes
0,000660,SK하이닉스,0,6,4,0,0,0,2,0,17,0,0,1.00,15,0,0,0,2750,
1,090430,아모레퍼시픽,0,12,0,14,0,0,4,0,45,0,0,0.96,26,0,0,0,6216,코스비전 기준 원재료와 상품도 포함했습니다. 합병 계약의 금액은 각 상대방의 개별 금액이 아니라 표에...
2,010770,평화홀딩스,0,1,1,0,0,0,0,0,1,0,0,1.00,1,0,0,0,224,
3,016610,DB증권,0,0,0,0,0,0,0,0,0,0,0,0.00,0,0,0,0,100,입력에 사업 관계나 제품을 식별할 수 있는 정보가 거의 없습니다.
4,279570,케이뱅크,0,1,1,0,0,0,0,0,1,0,0,1.00,1,0,0,0,437,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
960,016450,한세예스24홀딩스,0,20,13,1,26,0,3,4,81,0,0,0.99,36,0,0,0,8427,자동차 고객명 중 지엠 등은 구체적인 법인이 명시되지 않아 불확실하게 표시했다. 중국 지역 NBA 라...
961,011780,금호석유화학,0,35,2,4,1,0,0,0,58,0,0,1.00,39,0,0,0,7640,금호폴리켐의 주요 상표 칸은 상표들이 붙어 있어 분리하지 않았다. 시장점유율의 국내 시장 및 제품 범...
962,000670,영풍,0,14,7,12,0,4,6,0,69,0,0,0.57,65,0,0,0,8149,아연정광 매입처 표의 'Trafigura Glencore'는 별도 업체로 해석했습니다. TSL-1 공...
963,161890,한국콜마,0,19,0,10,8,2,42,0,124,0,0,0.95,95,0,0,0,12891,국내 ODM/OEM 시장점유율 약 40%대는 3개사 합계여서 개별 점유율로 기재하지 않았다. 데노수맙...


## 5. 결과 확인

**사용량**: 실행 도구가 알려 준 토큰 수다. Codex의 `input_tokens`는 캐시로 읽은 양(`cache_read_tokens`)을 포함한다. `cost_usd_equiv`는 같은 양을 API로 돌렸을 때의 요금 환산이라, 구독에서는 실제로 청구되지 않는다. 구독 한도는 Claude Code의 사용량 화면(5시간·주간 %)으로 따로 확인한다.

In [6]:
# ── 결과 확인: 상태별 건수, 토큰·시간·비용, 전체로 늘릴 때의 어림 ──
mine = runs[runs['prompt_hash'] == PROMPT_HASH]
if len(mine):
    tok = mine[mine['status'] == 'ok'][['input_tokens', 'cache_creation_tokens', 'cache_read_tokens', 'output_tokens', 'reasoning_tokens', 'duration_s', 'cost_usd_equiv']].astype(float)
    print('상태:', mine['status'].value_counts().to_dict())
    display(tok.describe().loc[['count', 'mean', '50%', 'max']].round(1))
    n_all = len(all_codes)
    per = tok.sum() / max(len(tok), 1)
    print(f"전체 {n_all}개로 늘리면 (회사당 평균 기준): 입력 약 {(per[['input_tokens', 'cache_creation_tokens']].sum() + (per['cache_read_tokens'] if RUNNER == 'claude' else 0)) * n_all / 1e6:.1f}백만 토큰,"
          f" 출력 약 {per['output_tokens'] * n_all / 1e6:.1f}백만 토큰")
else:
    print('아직 실행 기록이 없다.')

상태: {'ok': 965, 'limit': 2}


,input_tokens,cache_creation_tokens,cache_read_tokens,output_tokens,reasoning_tokens,duration_s,cost_usd_equiv
count,965.0,0.0,965.0,965.0,965.0,2.0,965.0
mean,7963.7,NaN,3171.6,4246.3,413.0,66.5,0.0
50%,7455.0,NaN,3017.0,3812.0,391.0,66.5,0.0
max,15955.0,NaN,15534.0,13945.0,1375.0,72.5,0.0


전체 1063개로 늘리면 (회사당 평균 기준): 입력 약 8.5백만 토큰, 출력 약 4.5백만 토큰


In [7]:
# ── 결과 확인: 회사 하나의 결과 ──
# 회사 하나의 결과 보기
SHOW = PILOT[0]
shown = ok_runs[ok_runs['stock_code'] == SHOW]
if len(shown):
    path = RAW / shown['raw_file'].iloc[-1]
    out = parse_result(json.loads(path.read_text(encoding='utf-8')))
    for key in ['products', 'brands', 'materials', 'customers', 'competitors', 'contracts']:
        rows = out.get(key, [])
        if key == 'materials':
            rows = [{'item': m['item'], 'name_en': m['name_en'], 'suppliers': ', '.join(s['name'] for s in m['suppliers'])} for m in rows]
        print(f'── {key} ({len(rows)})')
        if rows:
            display(pd.DataFrame(rows).drop(columns=['evidence', 'evidence_id'], errors='ignore'))
    print('notes:', out.get('notes'))
else:
    print(SHOW, '결과가 아직 없다.')

── products (26)


,name,name_en,category,segment,search_terms,end_markets,found_in,stage
0,TV,Television,영상가전,DX 부문,"[TV, television]",[],표,판매 중
1,모니터,Monitor,영상가전,DX 부문,"[computer monitor, display monitor]",[],표,판매 중
2,냉장고,Refrigerator,생활가전,DX 부문,"[refrigerator, fridge]",[],표,판매 중
3,세탁기,Washing machine,생활가전,DX 부문,"[washing machine, washer]",[],표,판매 중
4,에어컨,Air conditioner,생활가전,DX 부문,[air conditioner],[],표,판매 중
5,스마트폰,Smartphone,모바일 기기,DX 부문,"[smartphone, mobile phone]",[],표,판매 중
6,네트워크시스템,Network systems,통신장비,DX 부문,"[network systems, network equipment]",[],표,판매 중
7,PC,Personal computer,컴퓨터,DX 부문,"[PC, personal computer]",[],표,판매 중
8,DRAM,DRAM,메모리 반도체,DS 부문,"[DRAM, dynamic random access memory]",[],표,판매 중
9,NAND Flash,NAND flash,메모리 반도체,DS 부문,"[NAND flash, NAND memory]",[],표,판매 중


── brands (3)


,name,name_en,product,owner_type
0,Neo QLED,Neo QLED,TV,자사
1,더 프레임,The Frame,TV,자사
2,Q시리즈,Q Series,사운드바,자사


── materials (9)


,item,name_en,suppliers
0,모바일AP 솔루션,Mobile application processor solution,"Qualcomm, MediaTek"
1,디스플레이 패널,Display panel,"CSOT, SDP"
2,Camera Module,Camera module,"삼성전기㈜, 엠씨넥스"
3,Chemical,Chemicals,"솔브레인㈜, 동우화인켐㈜"
4,Wafer,Semiconductor wafer,"SILTRONIC, SK실트론㈜"
5,FPCA,Flexible printed circuit assembly,"㈜비에이치, SI FLEX"
6,Cover Glass,Cover glass,"Apple, LENS"
7,SOC(System-On-Chip),System-on-chip,"NVIDIA, Intel"
8,통신 모듈,Communication module,WNC(Wistron NeWeb Corp.)


── customers (9)


,name,name_en,name_sure,kind,product,share_text
0,SK텔레콤㈜,SK Telecom,True,판매 채널,,
1,㈜케이티,KT,True,판매 채널,,
2,㈜LG유플러스,LG Uplus,True,판매 채널,,
3,Alphabet,Alphabet,True,주요 고객,,주요 5대 매출처에 대한 매출비중은 전체 매출액 대비 약 15% 수준
4,Apple,Apple,True,주요 고객,,주요 5대 매출처에 대한 매출비중은 전체 매출액 대비 약 15% 수준
5,Deutsche Telekom,Deutsche Telekom,True,주요 고객,,주요 5대 매출처에 대한 매출비중은 전체 매출액 대비 약 15% 수준
6,Hong Kong Techtronics,Hong Kong Techtronics,False,주요 고객,,주요 5대 매출처에 대한 매출비중은 전체 매출액 대비 약 15% 수준
7,Supreme Electronics,Supreme Electronics,True,주요 고객,,주요 5대 매출처에 대한 매출비중은 전체 매출액 대비 약 15% 수준
8,"Tesla, Inc.","Tesla, Inc.",True,수주 상대,반도체 위탁생산,


── competitors (0)
── contracts (7)


,counterparty,name_en,name_sure,kind,subject,amount_text,period_text
0,"Tesla, Inc.","Tesla, Inc.",True,공급,반도체 위탁생산,"16,544 백만 달러",2025.07.26~2033.12.31
1,Google,Google,True,라이선스,상호 특허 라이선스,,
2,Google,Google,True,라이선스,"유럽 32개국 대상 Play Store, YouTube 등 구글 앱 사용",,
3,Ericsson,Ericsson,True,라이선스,상호 특허 라이선스,,
4,Qualcomm,Qualcomm,True,라이선스,상호 특허 라이선스ㆍ부제소,,
5,Huawei,Huawei,True,라이선스,상호 특허 라이선스,,
6,Nokia,Nokia,True,라이선스,상호 특허 라이선스,,


notes: HBM4 등 향후 공급 확대가 언급된 제품은 개발 여부나 현재 판매 여부가 명시되지 않아 단계가 불명입니다.
